<div style="text-align: center;">
    <hr>
    <h1 style="text-align: center;">Example Spark SQL</h1>
    <h3 style="text-align: center;">PROCESAMIENTO DE DATOS MASIVOS</h3>
    <p style="text-align: center;">Saul Razo Magallanes - 739974</p>
    <p style="text-align: center;">Ingeniería en Sistemas Computacionales</p>
    <p style="text-align: center;"><strong>Profesor:</strong> Pablo Camarillo Ramírez</p>
    <p style="text-align: center;"><strong>Fecha:</strong> 16/02/2026</p>
    <hr>
</div>


In [3]:
import findspark
findspark.init()

In [4]:
from spark_utils import SparkUtils

su = SparkUtils("Examples on Spark SQL", "spark://spark-master:7077")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/17 03:27:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
su._spark.sparkContext

<SparkContext master=spark://spark-master:7077 appName=Examples on Spark SQL>

In [7]:
data = [
    (1, "Alice", 29),
    (2, "Bob", 35),
    (3, "Charlie", 41)
]

df = su._spark.createDataFrame(data)
df.printSchema()

root
 |-- _1: long (nullable = true)
 |-- _2: string (nullable = true)
 |-- _3: long (nullable = true)



In [8]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [9]:
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("edad", IntegerType(), True)
])

df = su._spark.createDataFrame(data, schema)
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- edad: integer (nullable = true)



In [10]:
from datetime import datetime
from pyspark.sql.types import TimestampType, FloatType

factory_data = [
    ("M001", datetime(2026, 1, 26, 8, 0, 0), 75.3),
    ("M002", datetime(2026, 1, 26, 8, 5, 0), 68.7),
    ("M001", datetime(2026, 1, 26, 8, 10, 0), 76.1),
    ("M003", datetime(2026, 1, 26, 8, 15, 0), 72.4),
    ("M002", datetime(2026, 1, 26, 8, 20, 0), 69.8),
    ("M001", datetime(2026, 1, 26, 8, 25, 0), 77.5),
    ("M003", datetime(2026, 1, 26, 8, 30, 0), 73.2),
    ("M002", datetime(2026, 1, 26, 8, 35, 0), 70.1),
    ("M001", datetime(2026, 1, 26, 8, 40, 0), 78.0),
    ("M003", datetime(2026, 1, 26, 8, 45, 0), 74.6),
]

factory_schema = StructType([
    StructField("machine_id", StringType(), True),
    StructField("sensor_time", TimestampType(), True),
    StructField("temp", FloatType(), True)
])

df_factory = su._spark.createDataFrame(factory_data, factory_schema)
df_factory

DataFrame[machine_id: string, sensor_time: timestamp, temp: float]

In [11]:
from pyspark.sql.functions import col

filtered_df = df_factory.filter(col("temp") > 75)
filtered_df.show()
record_count = df_factory.count()
print(f"Total records: {record_count}")
filtered_record_count = filtered_df.count()
print(f"Total records: {filtered_record_count}")

+----------+-------------------+----+
|machine_id|        sensor_time|temp|
+----------+-------------------+----+
|      M001|2026-01-26 08:00:00|75.3|
|      M001|2026-01-26 08:10:00|76.1|
|      M001|2026-01-26 08:25:00|77.5|
|      M001|2026-01-26 08:40:00|78.0|
+----------+-------------------+----+

Total records: 10
Total records: 4


In [14]:
df_ordered = df_factory.orderBy(col("temp"), ascending = False)
df_ordered.show()

+----------+-------------------+----+
|machine_id|        sensor_time|temp|
+----------+-------------------+----+
|      M001|2026-01-26 08:40:00|78.0|
|      M001|2026-01-26 08:25:00|77.5|
|      M001|2026-01-26 08:10:00|76.1|
|      M001|2026-01-26 08:00:00|75.3|
|      M003|2026-01-26 08:45:00|74.6|
|      M003|2026-01-26 08:30:00|73.2|
|      M003|2026-01-26 08:15:00|72.4|
|      M002|2026-01-26 08:35:00|70.1|
|      M002|2026-01-26 08:20:00|69.8|
|      M002|2026-01-26 08:05:00|68.7|
+----------+-------------------+----+



In [20]:
df_groupped = df_factory.groupBy(col("machine_id")).count()
df_groupped.show()

[Stage 10:>                                                         (0 + 1) / 2]

+----------+-----+
|machine_id|count|
+----------+-----+
|      M002|    3|
|      M003|    3|
|      M001|    4|
+----------+-----+



In [23]:
from pyspark.sql.functions import avg, min, max

agg_df = df_factory.groupBy("machine_id").agg(
    avg("temp").alias("avg_temp"),
    min("temp").alias("min_temp"),
    max("temp").alias("max_temp")
)
agg_df.show()

+----------+-----------------+--------+--------+
|machine_id|         avg_temp|min_temp|max_temp|
+----------+-----------------+--------+--------+
|      M002|69.53333282470703|    68.7|    70.1|
|      M003|73.39999898274739|    72.4|    74.6|
|      M001|76.72500038146973|    75.3|    78.0|
+----------+-----------------+--------+--------+



In [28]:
df_ordered = df_factory.orderBy(col("temp"), ascending=False).limit(1)
df_ordered.show()

+----------+-------------------+----+
|machine_id|        sensor_time|temp|
+----------+-------------------+----+
|      M001|2026-01-26 08:40:00|78.0|
+----------+-------------------+----+

